In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import zarr
import tqdm
import shutil
import cupy as cp

In [ ]:
class MRI_Motion_Dataset(Dataset):
    def __init__(self, moving_zarr_path, fixed_zarr_path):
        super().__init__()
        self.moving_zarr_array = zarr.open(moving_zarr_path, mode='r')
        self.fixed_zarr_array = zarr.open(fixed_zarr_path, mode='r')
        assert self.moving_zarr_array.shape[3] == self.fixed_zarr_array.shape[3]
        self.num_images = self.moving_zarr_array.shape[3]
        self.original_shape = self.fixed_zarr_array.shape[:3] # H, W, D
        self.padded_shape = list(self.original_shape)
        for i in range(3):
            if self.padded_shape[i] % 4 != 0:
                self.padded_shape[i] = (self.padded_shape[i] // 4 + 1) * 4
        self.input_shape = (self.padded_shape[2], self.padded_shape[0], self.padded_shape[1]) # D, H, W
        print(f"Dataset initialisiert. Original H,W,D: {self.original_shape}. Padded H,W,D: {self.padded_shape}")

    def __len__(self):
        return self.num_images

    def __getitem__(self, idx):
        moving_np = self.moving_zarr_array[..., idx]
        fixed_np = self.fixed_zarr_array[..., idx]
        moving_vol = torch.from_numpy(moving_np.astype(np.float32)).permute(2,0,1).unsqueeze(0)
        fixed_vol = torch.from_numpy(fixed_np.astype(np.float32)).permute(2,0,1).unsqueeze(0)
        pad_d = self.padded_shape[2]-moving_vol.shape[1]
        pad_h = self.padded_shape[0]-moving_vol.shape[2]
        pad_w = self.padded_shape[1]-moving_vol.shape[3]
        padding = (pad_w//2, pad_w-pad_w//2, pad_h//2, pad_h-pad_h//2, pad_d//2, pad_d-pad_d//2)
        return F.pad(moving_vol, padding), F.pad(fixed_vol, padding)

class SpatialTransformer3D(nn.Module):
    def __init__(self, size):
        super().__init__()
        vectors = [torch.arange(0, s) for s in size]
        grids = torch.meshgrid(vectors, indexing='ij')
        grid = torch.stack(grids)
        self.register_buffer('grid', grid.unsqueeze(0).float(), persistent=False)

    def forward(self, src, flow):
        new_locs = self.grid + flow
        shape = flow.shape[2:]
        for i in range(len(shape)):
            new_locs[:, i, ...] = 2 * (new_locs[:, i, ...] / (shape[i] - 1) - 0.5)
        new_locs = new_locs.permute(0, 2, 3, 4, 1)[..., [2, 1, 0]]
        return F.grid_sample(src, new_locs, align_corners=True, padding_mode="border")

class UNet3D(nn.Module):
    def __init__(self, in_channels=2, out_channels=3):
        super().__init__()
        self.enc1 = self._conv_block(in_channels, 16)
        self.enc2 = self._conv_block(16, 32)
        self.pool = nn.MaxPool3d(2)
        self.bottleneck = self._conv_block(32, 64)
        self.upconv2 = nn.ConvTranspose3d(64, 32, kernel_size=2, stride=2)
        self.dec2 = self._conv_block(64, 32)
        self.upconv1 = nn.ConvTranspose3d(32, 16, kernel_size=2, stride=2)
        self.dec1 = self._conv_block(32, 16)
        self.final_conv = nn.Conv3d(16, out_channels, kernel_size=1)
        self.final_conv.weight.data.zero_()
        self.final_conv.bias.data.zero_()

    def _conv_block(self, in_c, out_c):
        return nn.Sequential(
            nn.Conv3d(in_c, out_c, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv3d(out_c, out_c, kernel_size=3, padding=1),
            nn.ReLU(inplace=True)
        )

    def forward(self, x_fixed, x_moving):
        x = torch.cat([x_fixed, x_moving], dim=1)
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        b = self.bottleneck(self.pool(e2))
        d2 = self.upconv2(b)
        if d2.shape[2:] != e2.shape[2:]:
            d2 = F.interpolate(d2, size=e2.shape[2:], mode='trilinear', align_corners=False)
        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.dec2(d2)
        d1 = self.upconv1(d2)
        if d1.shape[2:] != e1.shape[2:]:
            d1 = F.interpolate(d1, size=e1.shape[2:], mode='trilinear', align_corners=False)
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1)
        return self.final_conv(d1)

class RegistrationModel(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.registration_net = UNet3D()
        self.spatial_transformer = SpatialTransformer3D(size=input_size)

    def forward(self, moving, fixed):
        displacement_field = self.registration_net(fixed, moving)
        warped_moving = self.spatial_transformer(moving, displacement_field)
        return warped_moving, displacement_field

In [ ]:
MOVING_ZARR_PATH = "/mnt/dev_rep/repos/MRI-MoCoCo/DCE_codeset/DCE_codeset/MRI-Datasets/DCE/DCE"
FIXED_ZARR_PATH = "/mnt/dev_rep/repos/MRI-MoCoCo/DCE_codeset/MRI-Datasets/mdreg_DCE_fitting_results/coreg_zarr_2.zarr"
MODEL_PATH = "./best_registration_model.pth"
OUTPUT_WARPED_PATH = "final_warped_images.zarr"
OUTPUT_DVF_PATH = "final_displacement_fields.zarr"

BATCH_SIZE = 1

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Verwende Gerät: {device}")

full_dataset = MRI_Motion_Dataset(moving_zarr_path=MOVING_ZARR_PATH, fixed_zarr_path=FIXED_ZARR_PATH)
full_loader = DataLoader(full_dataset, batch_size=BATCH_SIZE, shuffle=False)

input_size = full_dataset.input_shape
model = RegistrationModel(input_size=input_size).to(device)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.eval() 
print(f"Modell von '{MODEL_PATH}' geladen.")

H, W, D = full_dataset.original_shape
T = len(full_dataset)

for path in [OUTPUT_WARPED_PATH, OUTPUT_DVF_PATH]:
    if os.path.exists(path):
        shutil.rmtree(path)

output_warped_zarr = zarr.open(OUTPUT_WARPED_PATH, mode='w',
                               shape=(H, W, D, T),
                               chunks=(H, W, D, 1),
                               dtype=full_dataset.moving_zarr_array.dtype)

output_dvf_zarr = zarr.open(OUTPUT_DVF_PATH, mode='w',
                            shape=(H, W, D, T, 3),
                            chunks=(H, W, D, 1, 3),
                            dtype='float32')

print(f"Output-Dateien erstellt:\n  - Warped: {OUTPUT_WARPED_PATH}\n  - DVF: {OUTPUT_DVF_PATH}")

current_index = 0
with torch.no_grad():
    progress_bar = tqdm.tqdm(full_loader, desc="Prozessiere gesamten Datensatz")
    for moving_batch, fixed_batch in progress_bar:
        moving_batch, fixed_batch = moving_batch.to(device), fixed_batch.to(device)

        warped_batch, displacement_batch = model(moving_batch, fixed_batch)

        warped_batch_cpu = warped_batch.cpu()
        displacement_batch_cpu = displacement_batch.cpu()
        batch_size_current = warped_batch_cpu.shape[0]

        for j in range(batch_size_current):
            warped_tensor = warped_batch_cpu[j]
            dvf_tensor = displacement_batch_cpu[j]
            
            pad_d = full_dataset.padded_shape[2] - D
            pad_h = full_dataset.padded_shape[0] - H
            pad_w = full_dataset.padded_shape[1] - W
            
            cropped_warped = warped_tensor[:, pad_d//2 : pad_d//2 + D, pad_h//2 : pad_h//2 + H, pad_w//2 : pad_w//2 + W]
            cropped_dvf = dvf_tensor[:, pad_d//2 : pad_d//2 + D, pad_h//2 : pad_h//2 + H, pad_w//2 : pad_w//2 + W]

            warped_np = cropped_warped.squeeze(0).numpy().transpose(1, 2, 0)
            dvf_np = cropped_dvf.permute(2, 3, 1, 0).numpy()

            idx_to_save = current_index + j
            output_warped_zarr[..., idx_to_save] = warped_np
            output_dvf_zarr[..., idx_to_save, :] = dvf_np

        current_index += batch_size_current

print(f"\nValidierung abgeschlossen.")
print(f"Die korrigierten Bilder wurden in '{OUTPUT_WARPED_PATH}' gespeichert.")
print(f"Die Displacement Fields wurden in '{OUTPUT_DVF_PATH}' gespeichert.")

In [ ]:
from sklearn.model_selection import train_test_split
from torch.utils.data import Subset

VALIDATION_SPLIT = 0.2
RANDOM_SEED = 42

full_dataset = MRI_Motion_Dataset(
    moving_zarr_path=MOVING_ZARR_PATH,
    fixed_zarr_path=FIXED_ZARR_PATH
)
print(f"Gesamte Datensatzgröße: {len(full_dataset)} Bilder")

all_indices = list(range(len(full_dataset)))

train_indices, val_indices = train_test_split(
    all_indices,
    test_size=VALIDATION_SPLIT,
    random_state=RANDOM_SEED,
    shuffle=True
)

train_dataset = Subset(full_dataset, train_indices)
val_dataset = Subset(full_dataset, val_indices)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"\nAufteilung mit sklearn abgeschlossen (Seed={RANDOM_SEED}):")
print(f"Größe des Trainings-Sets: {len(train_dataset)}")
print(f"Größe des Validierungs-Sets: {len(val_dataset)}")

Dataset initialisiert. Original H,W,D: (256, 256, 50). Padded H,W,D: [256, 256, 52]
Gesamte Datensatzgröße: 1000 Bilder

Aufteilung mit sklearn abgeschlossen (Seed=42):
Größe des Trainings-Sets: 800
Größe des Validierungs-Sets: 200


In [ ]:
MDREG_CORRECTED_PATH = "../DCE_codeset/MRI-Datasets/mdreg_DCE_fitting_results/coreg_zarr_2.zarr"
INDICES_LOAD_PATH = "val_indices_final.pth"
OUTPUT_MDREG_VALIDATION_PATH = "mdreg_corrected_validation_images.zarr"

try:
    val_indices_loaded = torch.load(INDICES_LOAD_PATH)
    print(f"Validierungs-Indizes erfolgreich aus '{INDICES_LOAD_PATH}' geladen.")
except FileNotFoundError:
    raise FileNotFoundError(f"FEHLER: Die Index-Datei '{INDICES_LOAD_PATH}' wurde nicht gefunden.")

mdreg_full_array = zarr.open(MDREG_CORRECTED_PATH, mode='r')
print(f"mdreg-Daten geladen mit Shape: {mdreg_full_array.shape}")

print("Selektiere Validierungsbilder aus dem mdreg-Datensatz...")
mdreg_validation_images = mdreg_full_array.oindex[..., val_indices_loaded]

if os.path.exists(OUTPUT_MDREG_VALIDATION_PATH):
    shutil.rmtree(OUTPUT_MDREG_VALIDATION_PATH)

print(f"Speichere das Subset in '{OUTPUT_MDREG_VALIDATION_PATH}'...")
output_zarr = zarr.open(
    OUTPUT_MDREG_VALIDATION_PATH,
    mode='w',
    shape=mdreg_validation_images.shape,
    chunks=(mdreg_full_array.chunks[0], mdreg_full_array.chunks[1], mdreg_full_array.chunks[2], 1),
    dtype=mdreg_validation_images.dtype
)
output_zarr[:] = mdreg_validation_images

print(f"\nErfolgreich! Die {len(val_indices_loaded)} mdreg-Validierungsbilder wurden in '{OUTPUT_MDREG_VALIDATION_PATH}' gespeichert.")

In [ ]:
MOVING_DATA_PATH = "/mnt/dev_rep/repos/MRI-MoCoCo/DCE_codeset/DCE_codeset/MRI-Datasets/DCE/DCE"
INDICES_LOAD_PATH = "val_indices_final.pth"
OUTPUT_MOVING_VALIDATION_PATH = "moving_validation_images.zarr"

try:
    val_indices_loaded = torch.load(INDICES_LOAD_PATH)
    print(f"Validierungs-Indizes erfolgreich aus '{INDICES_LOAD_PATH}' geladen.")
except FileNotFoundError:
    raise FileNotFoundError(f"FEHLER: Die Index-Datei '{INDICES_LOAD_PATH}' wurde nicht gefunden.")

moving_full_array = zarr.open(MOVING_DATA_PATH, mode='r')
print(f"Originale 'Moving'-Daten geladen mit Shape: {moving_full_array.shape}")

print("Selektiere die originalen Validierungsbilder...")
moving_validation_images = moving_full_array.oindex[..., val_indices_loaded]

if os.path.exists(OUTPUT_MOVING_VALIDATION_PATH):
    shutil.rmtree(OUTPUT_MOVING_VALIDATION_PATH)

print(f"Speichere das Subset in '{OUTPUT_MOVING_VALIDATION_PATH}'...")
output_zarr = zarr.open(
    OUTPUT_MOVING_VALIDATION_PATH,
    mode='w',
    shape=moving_validation_images.shape,
    chunks=(moving_full_array.chunks[0], moving_full_array.chunks[1], moving_full_array.chunks[2], 1),
    dtype=moving_validation_images.dtype
)
output_zarr[:] = moving_validation_images

print(f"\nErfolgreich! Die {len(val_indices_loaded)} originalen Validierungsbilder wurden in '{OUTPUT_MOVING_VALIDATION_PATH}' gespeichert.")

In [ ]:
MOVING_ZARR_PATH = "/mnt/dev_rep/repos/MRI-MoCoCo/DCE_codeset/DCE_codeset/MRI-Datasets/DCE/DCE"
FIXED_ZARR_PATH = "/mnt/dev_rep/repos/MRI-MoCoCo/DCE_codeset/MRI-Datasets/mdreg_DCE_fitting_results/coreg_zarr_2.zarr"
MODEL_PATH = "./best_registration_model.pth"
INDICES_LOAD_PATH = "val_indices_final.pth"
OUTPUT_WARPED_PATH = "validation_warped_only_val_images.zarr"
OUTPUT_DVF_PATH = "validation_displacement_fields.zarr"
BATCH_SIZE = 1

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Verwende Gerät: {device}")

full_dataset = MRI_Motion_Dataset(moving_zarr_path=MOVING_ZARR_PATH, fixed_zarr_path=FIXED_ZARR_PATH)

try:
    val_indices_loaded = torch.load(INDICES_LOAD_PATH)
    print(f"Validierungs-Indizes erfolgreich aus '{INDICES_LOAD_PATH}' geladen.")
except FileNotFoundError:
    raise FileNotFoundError(f"FEHLER: Die Index-Datei '{INDICES_LOAD_PATH}' wurde nicht gefunden.")

val_dataset = Subset(full_dataset, val_indices_loaded)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
print(f"Validierungs-Set mit {len(val_dataset)} Bildern ist bereit für die Inferenz.")

input_size = full_dataset.input_shape
model = RegistrationModel(input_size=input_size).to(device)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.eval()
print(f"Modell von '{MODEL_PATH}' geladen.")

H, W, D = full_dataset.original_shape
num_val_images = len(val_dataset)

for path in [OUTPUT_WARPED_PATH, OUTPUT_DVF_PATH]:
    if os.path.exists(path):
        shutil.rmtree(path)

output_warped_zarr = zarr.open(OUTPUT_WARPED_PATH, mode='w',
                               shape=(H, W, D, num_val_images),
                               chunks=(H, W, D, 1),
                               dtype=full_dataset.moving_zarr_array.dtype)

output_dvf_zarr = zarr.open(OUTPUT_DVF_PATH, mode='w',
                            shape=(H, W, D, num_val_images, 3),
                            chunks=(H, W, D, 1, 3),
                            dtype='float32')

print(f"Output-Dateien für {num_val_images} Validierungsbilder erstellt.")

current_index = 0
with torch.no_grad():
    progress_bar = tqdm.tqdm(val_loader, desc="Prozessiere Validierungs-Set")
    for moving_batch, fixed_batch in progress_bar:
        moving_batch, fixed_batch = moving_batch.to(device), fixed_batch.to(device)

        warped_batch, displacement_batch = model(moving_batch, fixed_batch)

        warped_batch_cpu = warped_batch.cpu()
        displacement_batch_cpu = displacement_batch.cpu()
        batch_size_current = warped_batch_cpu.shape[0]

        for j in range(batch_size_current):
            warped_tensor = warped_batch_cpu[j]
            dvf_tensor = displacement_batch_cpu[j]

            pad_d, pad_h, pad_w = full_dataset.padded_shape[2]-D, full_dataset.padded_shape[0]-H, full_dataset.padded_shape[1]-W
            cropped_warped = warped_tensor[:, pad_d//2:pad_d//2+D, pad_h//2:pad_h//2+H, pad_w//2:pad_w//2+W]
            cropped_dvf = dvf_tensor[:, pad_d//2:pad_d//2+D, pad_h//2:pad_h//2+H, pad_w//2:pad_w//2+W]

            warped_np = cropped_warped.squeeze(0).numpy().transpose(1, 2, 0)
            dvf_np = cropped_dvf.permute(2, 3, 1, 0).numpy()

            idx_to_save = current_index + j
            output_warped_zarr[..., idx_to_save] = warped_np
            output_dvf_zarr[..., idx_to_save, :] = dvf_np

        current_index += batch_size_current

print(f"\nValidierung abgeschlossen.")
print(f"Die {num_val_images} korrigierten Validierungsbilder wurden gespeichert in: '{OUTPUT_WARPED_PATH}'")
print(f"Die zugehörigen Displacement Fields wurden gespeichert in: '{OUTPUT_DVF_PATH}'")

In [24]:
mdreg_validation_images

array([[[[0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         ...,
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.]],

        [[0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         ...,
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.]],

        [[0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         ...,
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.]],

        ...,

        [[0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         ...,
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
    

In [16]:
slice = 24

In [18]:
coreg = mdreg_validation_images[:,:,slice,:]
coreg = cp.transpose(coreg, [2,1,0])

In [19]:
warped = output_warped_zarr[:,:,slice,:]
warped = cp.transpose(warped, [2,1,0])

In [22]:
original = moving_validation_images[:,:,slice,:]
original = cp.transpose(original, [2,1,0])

In [ ]:
from ipywidgets import interact, IntSlider
import matplotlib.pyplot as plt

def explore_3D_array_comparison_with_diff(
    arr1: np.ndarray, 
    arr2: np.ndarray, 
    arr3: np.ndarray, 
    cmap: str = 'gray'
):
    """
    Erstellt ein interaktives Widget, um Slices aus drei 3D-Arrays zu vergleichen.
    Zeigt die drei Bilder nebeneinander an und berechnet zusätzlich die absolute 
    Differenz zwischen dem zweiten (mittleren) und dritten (rechten) Bild.

    Args:
      arr1: Erstes 3D-Array (links), z.B. das Originalbild.
      arr2: Zweites 3D-Array (mitte), z.B. nach einer ersten Transformation.
      arr3: Drittes 3D-Array (rechts), z.B. nach einer zweiten Transformation.
      cmap: Farbkarte für die Darstellung der Bilder.
    """
    assert arr1.shape == arr2.shape == arr3.shape, "Alle drei Arrays müssen exakt die gleiche Form haben."

    diff_arr = np.absolute(arr2 - arr3)

    def fn(SLICE):
        fig, (ax1, ax2, ax3, ax4) = plt.subplots(1, 4, sharex='col', sharey='row', figsize=(20, 7))
        
        ax1.set_title('Image 1 (Left)', fontsize=15)
        ax1.imshow(arr1[SLICE, :, :], cmap=cmap)
        ax1.axis('off')

        ax2.set_title('Image 2 (Middle)', fontsize=15)
        ax2.imshow(arr2[SLICE, :, :], cmap=cmap)
        ax2.axis('off')

        ax3.set_title('Image 3 (Right)', fontsize=15)
        ax3.imshow(arr3[SLICE, :, :], cmap=cmap)
        ax3.axis('off')

        ax4.set_title('Difference (|Middle - Right|)', fontsize=15)
        im4 = ax4.imshow(diff_arr[SLICE, :, :], cmap='magma')
        ax4.axis('off')

        plt.tight_layout()
        plt.show()
    
    # Erstelle den interaktiven Slider
    interact(fn, SLICE=IntSlider(min=0, max=arr1.shape[0]-1, step=1, value=arr1.shape[0]//2, description='Slice:'))


In [ ]:
explore_3D_array_comparison_with_diff(original, coreg, warped)

interactive(children=(IntSlider(value=125, description='Slice:', max=249), Output()), _dom_classes=('widget-in…

In [38]:
import zarr
import numpy as np
from skimage.metrics import structural_similarity as ssim
from skimage.metrics import mean_squared_error as mse
import tqdm

In [ ]:
MDREG_VALIDATION_PATH = OUTPUT_MDREG_VALIDATION_PATH
MODEL_VALIDATION_PATH = OUTPUT_WARPED_PATH

def calculate_ncc(img1, img2):
    """Berechnet die Normalized Cross-Correlation zwischen zwei Bildern."""
    img1 = img1.astype(np.float64)
    img2 = img2.astype(np.float64)

    img1_norm = (img1 - img1.mean()) / img1.std()
    img2_norm = (img2 - img2.mean()) / img2.std()

    return np.mean(img1_norm * img2_norm)

try:
    mdreg_arr = zarr.open(MDREG_VALIDATION_PATH, mode='r')
    model_arr = zarr.open(MODEL_VALIDATION_PATH, mode='r')
except FileNotFoundError:
    raise FileNotFoundError("FEHLER: Einer der Zarr-Pfade wurde nicht gefunden. Stellen Sie sicher, dass die vorherigen Zellen ausgeführt wurden.")

assert mdreg_arr.shape == model_arr.shape, "Die Formen der beiden Datensätze stimmen nicht überein!"
num_images = mdreg_arr.shape[-1]
print(f"Vergleiche {num_images} Bilder aus den beiden Datensätzen.")

ssim_scores = []
ncc_scores = []
mse_scores = []

for i in tqdm.trange(num_images, desc="Berechne Metriken"):
    img_mdreg = mdreg_arr[..., i]
    img_model = model_arr[..., i]
    
    data_range = img_mdreg.max() - img_mdreg.min()
    current_ssim = ssim(img_mdreg, img_model, data_range=data_range, channel_axis=None)
    ssim_scores.append(current_ssim)
    
    current_ncc = calculate_ncc(img_mdreg, img_model)
    ncc_scores.append(current_ncc)
    
    current_mse = mse(img_mdreg, img_model)
    mse_scores.append(current_mse)

avg_ssim = np.mean(ssim_scores)
avg_ncc = np.mean(ncc_scores)
avg_mse = np.mean(mse_scores)

print("\n--- Durchschnittliche Metriken über das gesamte Validierungs-Set ---")
print(f"Structural Similarity (SSIM): {avg_ssim:.4f}")
print(f"Normalized Cross-Correlation (NCC): {avg_ncc:.4f}")
print(f"Mean Squared Error (MSE): {avg_mse:.4f}")
print("-------------------------------------------------------------------")

Vergleiche 250 Bilder aus den beiden Datensätzen.


Berechne Metriken: 100%|██████████| 250/250 [01:33<00:00,  2.68it/s]


--- Durchschnittliche Metriken über das gesamte Validierungs-Set ---
Structural Similarity (SSIM): 0.9995
Normalized Cross-Correlation (NCC): 0.9999
Mean Squared Error (MSE): 35.7094
-------------------------------------------------------------------


In [ ]:
MDREG_VALIDATION_PATH = OUTPUT_MOVING_VALIDATION_PATH
MODEL_VALIDATION_PATH = OUTPUT_WARPED_PATH

def calculate_ncc(img1, img2):
    """Berechnet die Normalized Cross-Correlation zwischen zwei Bildern."""
    img1 = img1.astype(np.float64)
    img2 = img2.astype(np.float64)
    
    img1_norm = (img1 - img1.mean()) / img1.std()
    img2_norm = (img2 - img2.mean()) / img2.std()

    return np.mean(img1_norm * img2_norm)

try:
    mdreg_arr = zarr.open(MDREG_VALIDATION_PATH, mode='r')
    model_arr = zarr.open(MODEL_VALIDATION_PATH, mode='r')
except FileNotFoundError:
    raise FileNotFoundError("FEHLER: Einer der Zarr-Pfade wurde nicht gefunden. Stellen Sie sicher, dass die vorherigen Zellen ausgeführt wurden.")

assert mdreg_arr.shape == model_arr.shape, "Die Formen der beiden Datensätze stimmen nicht überein!"
num_images = mdreg_arr.shape[-1]
print(f"Vergleiche {num_images} Bilder aus den beiden Datensätzen.")

ssim_scores = []
ncc_scores = []
mse_scores = []

for i in tqdm.trange(num_images, desc="Berechne Metriken"):
    img_mdreg = mdreg_arr[..., i]
    img_model = model_arr[..., i]
    
    data_range = img_mdreg.max() - img_mdreg.min()
    current_ssim = ssim(img_mdreg, img_model, data_range=data_range, channel_axis=None)
    ssim_scores.append(current_ssim)
    
    current_ncc = calculate_ncc(img_mdreg, img_model)
    ncc_scores.append(current_ncc)

    current_mse = mse(img_mdreg, img_model)
    mse_scores.append(current_mse)

avg_ssim = np.mean(ssim_scores)
avg_ncc = np.mean(ncc_scores)
avg_mse = np.mean(mse_scores)

print("\n--- Durchschnittliche Metriken über das gesamte Validierungs-Set ---")
print(f"Structural Similarity (SSIM): {avg_ssim:.4f}")
print(f"Normalized Cross-Correlation (NCC): {avg_ncc:.4f}")
print(f"Mean Squared Error (MSE): {avg_mse:.4f}")
print("-------------------------------------------------------------------")

Vergleiche 250 Bilder aus den beiden Datensätzen.


Berechne Metriken: 100%|██████████| 250/250 [01:30<00:00,  2.78it/s]


--- Durchschnittliche Metriken über das gesamte Validierungs-Set ---
Structural Similarity (SSIM): 0.9935
Normalized Cross-Correlation (NCC): 0.9976
Mean Squared Error (MSE): 644.5294
-------------------------------------------------------------------
